# 1. 다중선형회귀 실습 - 중고차 가격 예측

### 실습 조건
- 조건 1 (원핫인코딩): 범주형 변수인 Fuel을 원핫인코딩 하세요. 단, 다중공선성 함정을 피하기 위해 첫 번째 열은 삭제(drop_first=True)해야 합니다.
- 조건 2 (데이터 분리): 독립변수와 종속변수를 분리한 후, 훈련 데이터와 시험 데이터를 8:2 비율로 나누세요. (test_size=0.2, 결과 고정을 위해 random_state=42 설정)
- 조건 3 (모델 학습): LinearRegression 모델을 생성하고 훈련 데이터로 학습시키세요.
- 조건 4 (성별 및 수식 확인): 학습된 모델의 Y 절편(b)과 각 변수의 기울기(W)들을 출력해 보세요.
- 조건 5 (새로운 데이터 예측): 연식이 2021년, 주행거리가 40,000 Km, 연료 타입이 Gasoline인 중고차의 예측 가격을 출력하세요.

In [24]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# 데이터 생성
data = {
    'Year': [2020, 2018, 2022, 2019, 2021, 2017, 2023, 2018, 2020, 2022],    # 연식 (수치형)
    'Km': [50000, 80000, 20000, 95000, 35000, 120000, 10000, 85000, 60000, 25000], # 주행거리 (수치형)
    'Fuel': ['Gasoline', 'Diesel', 'Gasoline', 'Diesel', 'Gasoline', 'Diesel', 'Hybrid', 'Diesel', 'Hybrid', 'Gasoline'], # 연료 (범주형)
    'Price': [2500, 1500, 3500, 1200, 2900, 900, 4200, 1400, 2700, 3600]      # 가격 (타겟 Y, 단위: 만원)
}

dt = pd.DataFrame(data)

# X,y 분리
X = dt.drop(columns=['Price'])
y = dt['Price']

In [25]:
# 데이터 세트 분리
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [26]:
# 전처리 도구 정의
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(drop='first'),['Fuel'])],remainder='passthrough')

# 파이프라인 구축
pipeline = Pipeline(steps=[('preprocessor', ct),('model', LinearRegression())])

# 학습 (전처리와 모델 학습이 동시에 진행됨)
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [31]:
# 파이프라인 내부에서 최종 모델 객체 꺼내기
final_model = pipeline.named_steps['model']

print('=== 모델 수식 확인 ===')
print(f"y절편: {final_model.intercept_:.2f}")
print(f"기울기: {final_model.coef_}")

=== 모델 수식 확인 ===
y절편: -345182.59
기울기: [ 3.65422886e+02  7.94527363e+02  1.72388060e+02 -1.52985075e-02]


In [27]:
new = pd.DataFrame({
    'Year':[2021],
    'Km':[40000],
    'Fuel':['Gasoline']
})
pred = pipeline.predict(new)
print(f"예측 중고차 가격: {pred[0]:.2f}만 원")

예측 중고차 가격: 2967.16만 원


In [32]:
y_pred = pipeline.predict(X_test)

from sklearn.metrics import r2_score
score = r2_score(y_test, y_pred)
print(f"R²: {score:.4f}")

R²: 0.9330
